# MapleGuard — Banking Transaction Risk Lakehouse

**Author:** Hazim Ali  
**Goal:** demonstrate an evidence-led, privacy-safe banking data/AI workflow for Winter 2027 co-op recruiting.

> All records are deterministic and synthetic. Results are not suitable for real financial decisions.

## Decision context

Risk teams need a reliable path from raw transactions to monitored indicators and a prioritized human-review queue. This notebook reproduces the local pipeline; the Databricks source notebook builds the scalable Delta version.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))
from mapleguard import GenerationConfig, generate_transactions, train_evaluate_score, build_gold_tables

In [ ]:
transactions = generate_transactions(GenerationConfig())
transactions.head()

## Data contract

The generator creates anonymous IDs and behavioural features only. It includes explicit event time so the model can be evaluated on future-like data.

In [ ]:
transactions[['amount_cad','device_trust_score','account_age_days','transactions_24h','is_fraud']].describe().round(2)

## Time-aware model evaluation

Training, validation, and testing follow event time. The threshold is selected on validation data and reported metrics come from the untouched holdout period.

In [ ]:
result = train_evaluate_score(transactions)
result.metrics

In [ ]:
gold = build_gold_tables(result)
gold['gold_overview'].T

## Human-review queue

Alerts are ordered by model score and include non-causal behavioural reason codes. A production workflow would require case-management controls and investigator outcomes.

In [ ]:
gold['gold_alert_queue'].head(15)

## What this proves—and what it does not

The project proves reproducibility, time-aware validation, governed metric definitions, and decision-oriented communication. Synthetic performance does not prove real-world accuracy, fairness, privacy compliance, or production readiness.